In [1]:
import os
from pathlib import Path

import cv2
import numpy as np
import torch
import pycolmap
from PIL import Image

In [2]:
from pathlib import Path

parent_root = Path.cwd()                 # /parent
gs_root     = parent_root / "gaussian-splatting"
data_root   = gs_root / "models/images1/kitchen/"

model_dir = gs_root / "models/kitchen/"
images_dir  = data_root / "images"
sparse_dir  = data_root / "sparse/0"

print(images_dir)   # sanity check
print(images_dir.exists())



/orcd/home/002/yuanxiuw/guide-3d/gaussian-splatting/models/images1/kitchen/images
True


In [3]:
ply_path  = gs_root/ "models/kitchen/point_cloud/iteration_7000/point_cloud.ply"
output_dir     = data_root / "renders_3dgs"
output_dir.mkdir(exist_ok=True, parents=True)

In [4]:
recon = pycolmap.Reconstruction(sparse_dir)
images = list(recon.images.values())
images.sort(key=lambda im: im.image_id)  # deterministic order

print(f"Loaded {len(images)} registered images.")

Loaded 279 registered images.


In [5]:
repo_root = Path("gaussian-splatting").resolve()
sys.path.append(str(repo_root))
from gaussian_renderer import render
from scene import Scene, GaussianModel
from scene.cameras import Camera  # or whatever your repo uses

In [6]:
device = "cuda:0"

gaussians = GaussianModel(sh_degree=3)
gaussians.load_ply(str(ply_path)) 
bg = torch.tensor([1.0, 1.0, 1.0], device=device)

In [7]:
import sys, argparse
from pathlib import Path

import torch
import numpy as np
import cv2

repo_root = Path("gaussian-splatting").resolve()
sys.path.append(str(repo_root))

from arguments import ModelParams, PipelineParams
from scene import Scene, GaussianModel
from gaussian_renderer import render

device = "cuda:0"

# EDIT THESE TWO PATHS
source_path = data_root      # contains images/ and sparse/
model_path  = model_dir  # same as --model_path used in training

print("source_path:", source_path)
print("model_path:", model_path)

# --- build parser and args like in train.py ---
parser = argparse.ArgumentParser(description="Inference params")
lp = ModelParams(parser)
pp = PipelineParams(parser)

# minimal argv; all other options take defaults
argv = [
    "--source_path", str(source_path),
    "--model_path",  str(model_path),
    "--images",      "images",
]

args = parser.parse_args(argv)

dataset = lp.extract(args)   # same object they call "dataset" in training()
pipe    = pp.extract(args)

source_path: /orcd/home/002/yuanxiuw/guide-3d/gaussian-splatting/models/images1/kitchen
model_path: /orcd/home/002/yuanxiuw/guide-3d/gaussian-splatting/models/kitchen


In [8]:
gaussians = GaussianModel(dataset.sh_degree, optimizer_type="adam")  # optimizer_type only matters for training
scene = Scene(dataset, gaussians, load_iteration=7000, shuffle=False, resolution_scales=[1.0])

# Now scene.gaussians and scene.getTrainCameras() / getTestCameras() are ready
train_cams = scene.getTrainCameras(scale=1.0)   # list of Camera
test_cams  = scene.getTestCameras(scale=1.0)

print("train cams:", len(train_cams), "test cams:", len(test_cams))

Loading trained model at iteration 7000
Reading camera 279/279
Loading Training Cameras
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly specify '--resolution/-r' as 1
Loading Test Cameras
train cams: 279 test cams: 0


In [21]:
bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

In [24]:
import torch
import cv2
import numpy as np
from utils.loss_utils import l1_loss, ssim
from utils.image_utils import psnr

try:
    from fused_ssim import fused_ssim
    FUSED_SSIM_AVAILABLE = True
except ImportError:
    FUSED_SSIM_AVAILABLE = False


def render_metrics_frame(cam, gaussians, pipe, background, train_test_exp=False):
    """
    Render gaussians from `cam`, compute metrics vs cam.original_image,
    and return a labeled [GT | render] frame (H, 2W, 3, uint8) plus (L1, SSIM, PSNR).
    """

    # --- render (once) ---
    with torch.no_grad():
        pkg = render(cam, gaussians, pipe, background,
                     use_trained_exp=train_test_exp)
        img = torch.clamp(pkg["render"], 0.0, 1.0)  # (3,H,W)

    gt = torch.clamp(cam.original_image.to(img.device), 0.0, 1.0)

    # train_test_exp cropping if used
    if train_test_exp:
        img = img[..., img.shape[-1] // 2:]
        gt  = gt[...,  gt.shape[-1] // 2:]

    # alpha mask if present
    if cam.alpha_mask is not None:
        alpha = cam.alpha_mask.to(img.device)
        img = img * alpha
        gt  = gt  * alpha

    # --- metrics ---
    L1_val = l1_loss(img, gt).mean().double()

    if FUSED_SSIM_AVAILABLE:
        ssim_val = fused_ssim(img.unsqueeze(0), gt.unsqueeze(0)).double()
    else:
        ssim_val = ssim(img, gt).double()

    psnr_val = psnr(img, gt).mean().double()

    # --- convert to numpy for visualization ---
    img_vis = img.detach().cpu()   # (3,H,W)
    gt_vis  = gt.detach().cpu()

    if img_vis.ndim == 3 and img_vis.shape[0] == 3:
        img_vis = img_vis.permute(1, 2, 0)  # (H,W,3)
    if gt_vis.ndim == 3 and gt_vis.shape[0] == 3:
        gt_vis = gt_vis.permute(1, 2, 0)

    img_np = (img_vis.numpy() * 255.0).clip(0, 255).astype(np.uint8)
    gt_np  = (gt_vis.numpy()  * 255.0).clip(0, 255).astype(np.uint8)

    rend_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    gt_bgr   = cv2.cvtColor(gt_np,  cv2.COLOR_RGB2BGR)

    H, W = cam.image_height, cam.image_width
    rend_bgr = cv2.resize(rend_bgr, (W, H))
    gt_bgr   = cv2.resize(gt_bgr,   (W, H))

    frame = cv2.hconcat([gt_bgr, rend_bgr])

    # overlay text *after* concat
    name_text   = cam.image_name
    metric_text = f"SSIM {ssim_val.item():.3f}  PSNR {psnr_val.item():.2f} dB"

    cv2.putText(frame, name_text, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                (0, 255, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, metric_text, (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                (0, 255, 0), 2, cv2.LINE_AA)

    return frame, float(L1_val), float(ssim_val), float(psnr_val)

In [27]:
# Use resolution from first camera to configure VideoWriter
H0, W0 = train_cams[0].image_height, train_cams[0].image_width
fourcc = cv2.VideoWriter_fourcc(*"MJPG")
out_path = model_path / "comparison_debug.avi"
writer = cv2.VideoWriter(str(out_path), fourcc, 5, (2*W0, H0))
print("writer opened:", writer.isOpened())

for i, cam in enumerate(train_cams):
    frame, L1, ssim_val, psnr_val = render_metrics_frame(
        cam,
        scene.gaussians,
        pipe,
        background,
        train_test_exp=dataset.train_test_exp
    )

    # sanity: show pixel stats so we know frame content changes
    #print(f"{i}: {cam.image_name}, mean={frame.mean():.2f}, std={frame.std():.2f}, "
          #f"SSIM={ssim_val:.3f}, PSNR={psnr_val:.2f}")

    writer.write(frame)

writer.release()
print("wrote", out_path)

writer opened: True
0: DSCF0656.JPG, mean=122.07, std=61.38, SSIM=0.871, PSNR=23.23
1: DSCF0657.JPG, mean=121.09, std=61.94, SSIM=0.899, PSNR=24.97
2: DSCF0658.JPG, mean=119.65, std=62.32, SSIM=0.915, PSNR=27.12
3: DSCF0659.JPG, mean=120.35, std=61.55, SSIM=0.923, PSNR=28.82
4: DSCF0660.JPG, mean=121.64, std=61.59, SSIM=0.919, PSNR=28.10
5: DSCF0661.JPG, mean=123.98, std=61.22, SSIM=0.924, PSNR=27.14
6: DSCF0662.JPG, mean=123.81, std=62.21, SSIM=0.910, PSNR=25.51
7: DSCF0663.JPG, mean=124.73, std=61.94, SSIM=0.916, PSNR=24.83
8: DSCF0664.JPG, mean=121.07, std=61.15, SSIM=0.874, PSNR=21.00
9: DSCF0665.JPG, mean=125.14, std=61.35, SSIM=0.916, PSNR=24.42
10: DSCF0666.JPG, mean=125.88, std=62.22, SSIM=0.920, PSNR=26.60
11: DSCF0667.JPG, mean=127.76, std=61.64, SSIM=0.934, PSNR=28.50
12: DSCF0668.JPG, mean=127.16, std=61.59, SSIM=0.934, PSNR=28.35
13: DSCF0669.JPG, mean=126.67, std=60.56, SSIM=0.926, PSNR=28.51
14: DSCF0670.JPG, mean=126.92, std=59.88, SSIM=0.923, PSNR=27.73
15: DSCF0671.JP

In [26]:
print(L1)
print(ssim_val)
print(psnr_val)

0.030714761465787888
0.9102888107299805
27.487728118896484
